In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

# 02. SwiGLU Activation | 激活函数与门控机制 (SwiGLU Activation)

### ReLU（Rectified Linear Unit）

**核心公式：**
$$ \text{ReLU}(x) = \max(0, x) $$

---

### SiLU（Sigmoid Linear Unit）/ Swish

> **注：** SiLU 也叫 Swish，由 Google 提出（$\beta=1$ 时），是 Swish 的特例。

**核心公式：**
$$ \text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}} $$
其中 $\sigma(x)$ 是 Sigmoid 函数。

**函数特性：**
- 当 $x \to +\infty$ 时，SiLU$(x) \to x$（近似线性）；当 $x \to -\infty$ 时，SiLU$(x) \to 0$。
- **导数：** $\text{SiLU}'(x) = \sigma(x) + x \cdot \sigma(x) \cdot (1 - \sigma(x))$。

### GLU 与 SwiGLU

#### 1. GLU（Gated Linear Unit）

**核心思想：**  
传统 MLP 的前馈计算为 $W_{down}(\sigma(W_{up}x))$，即“激活 → 线性变换”。  
GLU 引入**门控机制**，将输入分成两条路径：

- **门控路径（Gate）**：经过激活函数，控制信息流动的“开关”。
- **线性路径（Value）**：保持线性变换，作为待传递的内容。

两条路径逐元素相乘（Hadamard Product），再经过输出投影。

**公式：**
$$ \text{GLU}(x, W_1, W_2, W_3) = (xW_1 \odot \sigma(xW_2)) W_3 $$

---

#### 2. SwiGLU（Swish-Gated Linear Unit）

**核心改进：**  
将 GLU 中的激活函数 $\sigma$ 替换为 **Swish**（即 $x \cdot \text{Sigmoid}(\beta x)$）。

**公式：**
$$ \text{SwiGLU}(x, W_1, W_2, W_3) = (xW_1 \odot \text{Swish}(xW_2)) W_3 $$

其中 $\text{Swish}(z) = z \cdot \sigma(z) = \frac{z}{1 + e^{-z}}$（PyTorch 中 $\beta=1$ 时等同于 `SiLU`）。

#### 3. 与传统 MLP 的结构对比

| 结构 | 公式 | 参数量 |
|------|------|--------|
| 标准 MLP（ReLU） | $W_3 \cdot \text{ReLU}(W_1 x)$ | $2dh$ |
| GLU | $(W_1 x \odot \sigma(W_2 x)) W_3$ | $3dh$（多一个门控矩阵） |
| SwiGLU | $(W_1 x \odot \text{Swish}(W_2 x)) W_3$ | $3dh$（同 GLU） |

### 参数量对齐

**典型的面试问题：**
> “在 GPT-2 中，隐藏层维度通常是输入维度 $d$ 的 4 倍（即 $4d$）。但在使用 SwiGLU 的 LLaMA 中，为什么隐藏层维度变成了 $\frac{8}{3}d$ 并向上取整？”

**推导过程：**
1. **标准 MLP 参数量**：
   输入为 $d$，隐藏层为 $h$。
   有两个投影矩阵（升维 $d \to h$，降维 $h \to d$）。
   总参数量 = $2 \cdot (d \times h)$。
   当 $h = 4d$ 时，总参数量 = $2 \cdot 4d^2 = \mathbf{8d^2}$。

2. **SwiGLU MLP 参数量**：
   输入为 $d$，隐藏层为 $h_{swiglu}$。
   因为有**门控机制**，升维阶段需要**两个**投影矩阵（$W_{gate}$ 和 $W_{up}$，均是 $d \to h_{swiglu}$）。
   降维阶段需要**一个**矩阵（$W_{down}$，是 $h_{swiglu} \to d$）。
   总参数量 = $3 \cdot (d \times h_{swiglu})$。

3. **对齐参数量**：
   为了使得 SwiGLU 的计算开销（参数量）与原始模型完全相同：
   $3 \cdot d \cdot h_{swiglu} = 8d^2$
   解得：$h_{swiglu} = \mathbf{\frac{8}{3}d}$
   
这正是 LLaMA 源码中对中间层维度进行 `int(8 * hidden_size / 3)` 计算的根本原因。

### 工业级实现框架与性能陷阱 (Memory Bound)

**性能陷阱 1：张量并行 (TP) 与内存对齐**
在真实的 LLaMA 源码中，除了按 $8/3$ 计算出隐藏层维度，还需要将其向上取整对齐到一个 `multiple_of`（通常是 256）的倍数。
这不仅是为了让单卡 Tensor Core（通常要求 8-byte 或 32-byte 对齐）跑得更快，更是因为大模型训练会使用**张量并行 (Tensor Parallelism)**。如果隐藏层维度不能被 GPU 数量整除（例如 $TP=8$ 时，256 的倍数分给 8 张卡，每张卡至少能分到 32 维），在切分权重矩阵时就会发生严重的报错。

**性能陷阱 2：访存瓶颈 (Memory Bound) 与矩阵融合**
在最朴素的代码实现中，开发者会分别定义并执行 `gate_proj(x)` 和 `up_proj(x)`。
由于这两个线性层**共享完全相同的输入张量 $x$**，分开计算会导致巨大的输入 $x$ 被 GPU 从全局显存 (HBM) 中读取两次。
> **工业界解法 (Matrix Fusion)**：
> 在 vLLM、Megatron 等主流框架中，标准的做法是将 $W_{gate}$ 和 $W_{up}$ 这两个形状为 `[hidden_size, intermediate_size]` 的权重矩阵，在初始化时拼接成一个**巨大的融合矩阵 `gate_up_proj`**，其形状为 `[hidden_size, 2 * intermediate_size]`。
> 
> 在前向传播时，输入 $x$ 只需要被读取一次进行一次大规模矩阵乘法。得到的结果再通过 `torch.chunk(2, dim=-1)` 切分为两半，分别作为 gate 和 up 块。这极大地缓解了内存带宽瓶颈。

In [2]:
def calculate_intermediate_size(hidden_size: int, multiple_of: int = 256):
    """
    计算 LLaMA 风格的 SwiGLU 隐藏层维度
    
    规则：
    1. 取 hidden_size 的 8/3
    2. 为了硬件对齐（如 Tensor Core），通常要求是 multiple_of 的倍数。
       因此将结果除以 multiple_of，向上取整后再乘以 multiple_of。
    """
    # ==========================================
    # TODO 1: 计算理论隐藏层大小 (8/3 * hidden_size)
    # 提示: 注意使用整数除法
    # ==========================================
    intermediate_size = int(hidden_size * 8 / 3)
    
    # ==========================================
    # TODO 2: 向 multiple_of 对齐 (向上取整)
    # 提示: 思考如何利用整除的特性实现向上取整
    # ==========================================
    aligned_size = ((intermediate_size + multiple_of - 1) // multiple_of) * multiple_of
    
    return aligned_size
    pass

class SwiGLU_MLP(nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        # ==========================================
        # TODO 3: 定义工业级 SwiGLU 的投影矩阵
        # ==========================================
        self.gate_up_proj = nn.Linear(hidden_size, 2 * intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 4: 组装工业级 SwiGLU 前向传播
        # ==========================================
        gate_up = self.gate_up_proj(x)
        gate, up = torch.chunk(gate_up, 2, dim=-1)
        return self.down_proj(F.silu(gate) * up)
        pass